In [30]:
from pyscf import gto, scf, dft, lib
import numpy as np

## Setups

In [2]:
mol = gto.Mole(atom="O; H 1 0.94; H 1 0.94 2 104.5", basis="def2-TZVP").build()

In [3]:
mf = scf.RKS(mol, xc="TPSS0").run()

converged SCF energy = -76.4430874743457


In [4]:
arrays = {
    "mo_coeff": np.asarray(mf.mo_coeff, order="C"),
    "mo_occ": np.asarray(mf.mo_occ, order="C"),
    "rdm1": mf.make_rdm1(),
    "coords": mf.grids.coords,
    "weights": mf.grids.weights,
}

In [5]:
mo_coeff = np.asarray(mf.mo_coeff, order="C")
mo_occ = np.asarray(mf.mo_occ, order="C")
rdm1 = mf.make_rdm1()
coords = mf.grids.coords
weights = mf.grids.weights

In [6]:
grids = mf.grids

In [7]:
ni = dft.numint.NumInt()

In [8]:
np.savez("h2o", **arrays)

## Get rho by dm

In [12]:
ao = ni.eval_ao(mol, grids.coords, deriv=2)
rho_rho = ni.eval_rho(mol, ao[0], rdm1, xctype="lda")
rho_sigma = ni.eval_rho(mol, ao[0:4], rdm1, xctype="gga")
rho_tau = ni.eval_rho(mol, ao[0:4], rdm1, xctype="mgga", with_lapl=False)
rho_lapl = ni.eval_rho(mol, ao[0:10], rdm1, xctype="mgga", with_lapl=True)

In [13]:
rho_rho.shape, lib.fp(rho_rho)

((33704,), np.float64(-438.0303348067834))

In [14]:
rho_sigma.shape, lib.fp(rho_sigma)

((4, 33704), np.float64(25704.14480085445))

In [15]:
rho_tau.shape, lib.fp(rho_tau)

((5, 33704), np.float64(17140.300791589947))

In [16]:
rho_lapl_ = rho_lapl[[0, 1, 2, 3, 5, 4]]

In [18]:
rho_lapl_.shape, lib.fp(rho_lapl_[0:5]), lib.fp(rho_lapl_[5])

((6, 33704), np.float64(17140.300791589947), np.float64(2470300.1875723572))